# TASK 1 — DATA CLEANING, MERGE & ANALYSIS

## 1. Load Data

In [15]:
# 1. Load and Detect Workbook

from google.colab import files
import pandas as pd
import io

try:
    uploaded = files.upload()

    if len(uploaded) != 1:
        raise ValueError(
            "Please upload exactly one Excel workbook."
        )

    file_name = next(iter(uploaded))

    if not file_name.lower().endswith(".xlsx"):
        raise ValueError(
            "Please upload an .xlsx file."
        )

    excel_file = io.BytesIO(
        uploaded[file_name]
    )

    # Detect sheets automatically
    excel_book = pd.ExcelFile(excel_file)

    detected_sheets = excel_book.sheet_names

    print(
        f"DONE - Workbook '{file_name}' loaded successfully."
    )

    print("\nINFO - Sheets detected:")

    for sheet in detected_sheets:
        print(f"  - {sheet}")

    # Required source sheets
    required_sheets = [
        "Rank_Tracker_Export",
        "GSC_Query_Export",
        "Keyword_Category_Map"
    ]

    missing_sheets = [
        sheet
        for sheet in required_sheets
        if sheet not in detected_sheets
    ]

    if missing_sheets:
        raise ValueError(
            "Missing required sheet(s): "
            + ", ".join(missing_sheets)
        )

    print(
        "\nDONE - All required source sheets were found."
    )

    # Load required sheets into internal variables
    rank_tracker = pd.read_excel(
        excel_book,
        sheet_name="Rank_Tracker_Export"
    )

    gsc = pd.read_excel(
        excel_book,
        sheet_name="GSC_Query_Export"
    )

    category_map = pd.read_excel(
        excel_book,
        sheet_name="Keyword_Category_Map"
    )

    print(
        "DONE - Required sheets loaded into the workflow."
    )

except Exception as e:
    print(
        f"FAIL - Workbook loading error: {e}"
    )

Saving supply_chain_seo_export.xlsx to supply_chain_seo_export (2).xlsx
DONE - Workbook 'supply_chain_seo_export (2).xlsx' loaded successfully.

INFO - Sheets detected:
  - Rank_Tracker_Export
  - GSC_Query_Export
  - Keyword_Category_Map

DONE - All required source sheets were found.
DONE - Required sheets loaded into the workflow.


## 2. Validate Input Structure

In [6]:
# 2. Validate Input Structure

try:
    required_columns = {
        "Rank_Tracker_Export": [
            "Week",
            "Keyword",
            "URL",
            "Device",
            "Rank",
            "Est. Search Volume",
            "Clicks",
            "Impressions"
        ],
        "GSC_Query_Export": [
            "Month",
            "Query",
            "Clicks",
            "Impressions",
            "CTR",
            "Avg Position"
        ],
        "Keyword_Category_Map": [
            "Keyword",
            "Theme",
            "Priority"
        ]
    }

    datasets = {
        "Rank_Tracker_Export": rank_tracker,
        "GSC_Query_Export": gsc,
        "Keyword_Category_Map": category_map
    }

    print("INFO - Columns detected:\n")

    for sheet_name, df in datasets.items():
        print(f"{sheet_name}:")
        for column in df.columns:
            print(f"  - {column}")
        print()

    missing_columns = {}

    for sheet_name, required in required_columns.items():

        existing_columns = datasets[
            sheet_name
        ].columns.tolist()

        missing = [
            column
            for column in required
            if column not in existing_columns
        ]

        if missing:
            missing_columns[
                sheet_name
            ] = missing

    if missing_columns:

        error_messages = []

        for sheet_name, columns in missing_columns.items():

            error_messages.append(
                f"{sheet_name}: "
                + ", ".join(columns)
            )

        raise ValueError(
            "Missing required column(s):\n"
            + "\n".join(error_messages)
        )

    print(
        "DONE - Input structure validated successfully."
    )

except Exception as e:
    print(
        f"FAIL - Input structure validation error: {e}"
    )

INFO - Columns detected:

Rank_Tracker_Export:
  - Week
  - Keyword
  - URL
  - Device
  - Rank
  - Est. Search Volume
  - Clicks
  - Impressions

GSC_Query_Export:
  - Month
  - Query
  - Clicks
  - Impressions
  - CTR
  - Avg Position

Keyword_Category_Map:
  - Keyword
  - Theme
  - Priority

DONE - Input structure validated successfully.


## 3. Clean and Standardize Keywords

In [16]:
# 3. Clean and Standardize Keywords

try:
    # Keep null values as null and only clean real text values
    rank_tracker["Keyword_Clean"] = (
        rank_tracker["Keyword"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    gsc["Keyword_Clean"] = (
        gsc["Query"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    category_map["Keyword_Clean"] = (
        category_map["Keyword"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    # Remove exact duplicate rows from Rank Tracker
    rows_before = len(rank_tracker)

    rank_tracker = (
        rank_tracker
        .drop_duplicates()
        .copy()
    )

    duplicates_removed = (
        rows_before - len(rank_tracker)
    )

    print(
        "DONE - Keywords standardized successfully."
    )

    print(
        f"INFO - {duplicates_removed} exact duplicate "
        f"Rank Tracker row(s) removed."
    )

except Exception as e:
    print(
        f"FAIL - Keyword cleaning error: {e}"
    )

DONE - Keywords standardized successfully.
INFO - 6 exact duplicate Rank Tracker row(s) removed.


## 4. Data Quality Checks

In [17]:
# 4. Data Quality Checks

try:
    issues_found = False

    datasets = {
        "Rank Tracker": (rank_tracker, "Keyword_Clean"),
        "GSC": (gsc, "Keyword_Clean"),
        "Category Mapping": (category_map, "Keyword_Clean")
    }

    # Check missing or empty keywords
    for name, (df, keyword_col) in datasets.items():

        missing_keywords = df[keyword_col].isna().sum()

        empty_keywords = (
            df[keyword_col]
            .fillna("")
            .str.strip()
            .eq("")
            .sum()
        )

        if missing_keywords > 0:
            print(
                f"WARNING - {name}: "
                f"{missing_keywords} missing keyword(s) found."
            )
            issues_found = True

        if empty_keywords > 0:
            print(
                f"WARNING - {name}: "
                f"{empty_keywords} empty keyword(s) found."
            )
            issues_found = True

    # Check if one cleaned keyword appears more than once
    # in the Category Mapping
    duplicate_mapping = (
        category_map["Keyword_Clean"]
        .notna()
        & category_map["Keyword_Clean"]
        .duplicated(keep=False)
    )

    if duplicate_mapping.any():

        duplicated_keywords = (
            category_map.loc[
                duplicate_mapping,
                "Keyword_Clean"
            ]
            .nunique()
        )

        print(
            "WARNING - Category Mapping: "
            f"{duplicated_keywords} duplicated cleaned "
            "keyword(s) found."
        )

        display(
            category_map.loc[
                duplicate_mapping,
                [
                    "Keyword",
                    "Keyword_Clean",
                    "Theme",
                    "Priority"
                ]
            ].sort_values("Keyword_Clean")
        )

        issues_found = True

    # Final result
    if issues_found:
        print(
            "\nWARNING - Data quality checks completed. "
            "Review the issues above before continuing."
        )
    else:
        print(
            "DONE - Data quality checks completed. "
            "No additional issues found."
        )

except Exception as e:
    print(
        f"FAIL - Data quality check error: {e}"
    )

DONE - Data quality checks completed. No additional issues found.


## 5. Standardize Dates

In [18]:
# 5. Standardize Rank Tracker Dates

try:
    rank_tracker = rank_tracker.copy()

    converted_week = pd.to_datetime(
        rank_tracker["Week"],
        format="mixed",
        errors="coerce"
    )

    failed_dates = converted_week.isna().sum()

    if failed_dates > 0:
        raise ValueError(
            f"{failed_dates} date(s) could not be converted."
        )

    rank_tracker["Week"] = converted_week

    rank_tracker["Month_Key"] = (
        converted_week.dt.strftime("%Y-%m")
    )

    print(
        "DONE - All Rank Tracker dates were standardized successfully."
    )

except Exception as e:
    print(
        f"FAIL - Error during date standardization: {e}"
    )

DONE - All Rank Tracker dates were standardized successfully.


## 6. Prepare GSC Data

In [19]:
# 6. Prepare GSC Data

try:
    gsc = gsc.copy()

    gsc["Month_Key"] = pd.to_datetime(
        gsc["Month"],
        format="%B %Y",
        errors="coerce"
    ).dt.strftime("%Y-%m")

    failed_months = gsc["Month_Key"].isna().sum()

    if failed_months > 0:
        raise ValueError(
            f"{failed_months} month value(s) could not be converted."
        )

    # Recalculate CTR from source metrics
    gsc["GSC_CTR"] = (
        gsc["Clicks"] / gsc["Impressions"]
    ).where(gsc["Impressions"] > 0)

    gsc["Merge_Key"] = (
        gsc["Month_Key"]
        + "|"
        + gsc["Keyword_Clean"]
    )

    duplicate_keys = (
        gsc["Merge_Key"]
        .dropna()
        .duplicated()
        .sum()
    )

    if duplicate_keys > 0:
        raise ValueError(
            f"{duplicate_keys} duplicate Month + Keyword key(s) found."
        )

    print(
        "DONE - GSC data prepared successfully."
    )

    print(
        "INFO - All Month + Keyword keys are unique."
    )

except Exception as e:
    print(
        f"FAIL - Error during GSC preparation: {e}"
    )

DONE - GSC data prepared successfully.
INFO - All Month + Keyword keys are unique.


## 7. Aggregate Rank Tracker

In [20]:
## 7. Aggregate Rank Tracker Data

try:
    rank_tracker_monthly = (
        rank_tracker
        .groupby(
            ["Month_Key", "Keyword_Clean"],
            as_index=False
        )
        .agg({
            "Rank": "mean",
            "Est. Search Volume": "mean",
            "Clicks": "sum",
            "Impressions": "sum"
        })
        .rename(columns={
            "Rank": "Avg_Rank",
            "Est. Search Volume": "Avg_Search_Volume",
            "Clicks": "Rank_Tracker_Clicks",
            "Impressions": "Rank_Tracker_Impressions"
        })
    )

    duplicate_keys = (
        rank_tracker_monthly[
            ["Month_Key", "Keyword_Clean"]
        ]
        .duplicated()
        .sum()
    )

    if duplicate_keys > 0:
        raise ValueError(
            f"{duplicate_keys} duplicate Month + Keyword key(s) remain."
        )

    print(
        f"DONE - Rank Tracker aggregated successfully "
        f"into {len(rank_tracker_monthly)} Month + Keyword rows."
    )

except Exception as e:
    print(
        f"FAIL - Error during Rank Tracker aggregation: {e}"
    )

DONE - Rank Tracker aggregated successfully into 54 Month + Keyword rows.


## 8. Merge Data Sources

In [21]:
# 8. Merge Data Sources

try:
    rank_tracker_monthly = rank_tracker_monthly.copy()

    # Create Month + Keyword key
    rank_tracker_monthly["Merge_Key"] = (
        rank_tracker_monthly["Month_Key"]
        + "|"
        + rank_tracker_monthly["Keyword_Clean"]
    )

    # Prepare GSC fields
    gsc_merge = gsc[
        [
            "Merge_Key",
            "Clicks",
            "Impressions",
            "GSC_CTR",
            "Avg Position"
        ]
    ].rename(columns={
        "Clicks": "GSC_Clicks",
        "Impressions": "GSC_Impressions",
        "Avg Position": "GSC_Avg_Position"
    })

    # Merge Rank Tracker with GSC
    merged_data = rank_tracker_monthly.merge(
        gsc_merge,
        on="Merge_Key",
        how="left"
    )

    # Prepare Category Mapping
    category_merge = category_map[
        [
            "Keyword_Clean",
            "Theme",
            "Priority"
        ]
    ].drop_duplicates(
        subset=["Keyword_Clean"]
    )

    # Add Theme and Priority
    merged_data = merged_data.merge(
        category_merge,
        on="Keyword_Clean",
        how="left"
    )

    # Flag missing mappings
    merged_data["Theme"] = (
        merged_data["Theme"]
        .fillna("Unmapped")
    )

    merged_data["Priority"] = (
        merged_data["Priority"]
        .fillna("Unmapped")
    )

    merged_data["Unmapped_Flag"] = (
        merged_data["Theme"]
        .eq("Unmapped")
        .map({
            True: "Yes",
            False: "No"
        })
    )

    # Validation
    duplicate_keys = (
        merged_data["Merge_Key"]
        .duplicated()
        .sum()
    )

    if duplicate_keys > 0:
        raise ValueError(
            f"{duplicate_keys} duplicate Merge_Key(s) "
            "were created during the merge."
        )

    missing_gsc = (
        merged_data["GSC_Impressions"]
        .isna()
        .sum()
    )

    unmapped_keywords = (
        merged_data.loc[
            merged_data["Unmapped_Flag"] == "Yes",
            "Keyword_Clean"
        ]
        .drop_duplicates()
        .nunique()
    )

    print(
        f"DONE - Data sources merged successfully "
        f"into {len(merged_data)} rows."
    )

    print(
        f"INFO - {missing_gsc} row(s) did not find "
        "a matching GSC record."
    )

    print(
        f"INFO - {unmapped_keywords} unique keyword(s) "
        "do not have a Product Theme mapping."
    )

except Exception as e:
    print(
        f"FAIL - Error during data merge: {e}"
    )

DONE - Data sources merged successfully into 54 rows.
INFO - 0 row(s) did not find a matching GSC record.
INFO - 3 unique keyword(s) do not have a Product Theme mapping.


## 9. Validate Merged Data

In [22]:
# 9. Validate Merged Data

try:
    duplicate_keys = merged_data["Merge_Key"].duplicated().sum()
    missing_keys = merged_data["Merge_Key"].isna().sum()

    unmapped_keywords = (
        merged_data.loc[
            merged_data["Unmapped_Flag"] == "Yes",
            "Keyword_Clean"
        ]
        .drop_duplicates()
        .tolist()
    )

    if duplicate_keys > 0:
        raise ValueError(
            f"{duplicate_keys} duplicate Merge_Key(s) found."
        )

    if missing_keys > 0:
        raise ValueError(
            f"{missing_keys} missing Merge_Key(s) found."
        )

    print(
        "DONE - Merged dataset validated successfully."
    )

    print(
        f"INFO - {len(unmapped_keywords)} "
        "unique unmapped keyword(s) found."
    )

    if unmapped_keywords:
        for keyword in unmapped_keywords:
            print(f"  - {keyword}")

except Exception as e:
    print(
        f"FAIL - Merged data validation error: {e}"
    )

DONE - Merged dataset validated successfully.
INFO - 3 unique unmapped keyword(s) found.
  - logistics automation software
  - supply chain risk management
  - third party logistics software


## 10. Theme Visibility Analysis

In [23]:
# 10. Theme Visibility Analysis

try:
    # Detect first and last available month
    available_months = sorted(
        merged_data["Month_Key"]
        .dropna()
        .unique()
    )

    if len(available_months) < 2:
        raise ValueError(
            "At least two months are required for visibility analysis."
        )

    first_month = available_months[0]
    last_month = available_months[-1]

    # Monthly performance by Theme
    theme_monthly = (
        merged_data
        .groupby(
            ["Month_Key", "Theme"],
            as_index=False
        )
        .agg(
            Avg_Rank=("Avg_Rank", "mean"),
            GSC_Impressions=("GSC_Impressions", "sum"),
            GSC_Clicks=("GSC_Clicks", "sum")
        )
    )

    # First available month
    first_period = (
        theme_monthly[
            theme_monthly["Month_Key"] == first_month
        ]
        .drop(columns="Month_Key")
        .rename(columns={
            "Avg_Rank": "First_Avg_Rank",
            "GSC_Impressions": "First_GSC_Impressions",
            "GSC_Clicks": "First_GSC_Clicks"
        })
    )

    # Last available month
    last_period = (
        theme_monthly[
            theme_monthly["Month_Key"] == last_month
        ]
        .drop(columns="Month_Key")
        .rename(columns={
            "Avg_Rank": "Last_Avg_Rank",
            "GSC_Impressions": "Last_GSC_Impressions",
            "GSC_Clicks": "Last_GSC_Clicks"
        })
    )

    theme_visibility = first_period.merge(
        last_period,
        on="Theme",
        how="outer"
    )

    # Safe percentage change
    def percentage_change(first_value, last_value):

        if (
            pd.isna(first_value)
            or pd.isna(last_value)
            or first_value == 0
        ):
            return pd.NA

        return (
            (last_value - first_value)
            / first_value
            * 100
        )

    theme_visibility["Impressions_Change"] = (
        theme_visibility.apply(
            lambda row: percentage_change(
                row["First_GSC_Impressions"],
                row["Last_GSC_Impressions"]
            ),
            axis=1
        )
    )

    theme_visibility["Clicks_Change"] = (
        theme_visibility.apply(
            lambda row: percentage_change(
                row["First_GSC_Clicks"],
                row["Last_GSC_Clicks"]
            ),
            axis=1
        )
    )

    # Visibility classification
    def classify_visibility(row):

        first_rank = row["First_Avg_Rank"]
        last_rank = row["Last_Avg_Rank"]
        first_impressions = row["First_GSC_Impressions"]
        last_impressions = row["Last_GSC_Impressions"]

        if (
            pd.isna(first_rank)
            or pd.isna(last_rank)
            or pd.isna(first_impressions)
            or pd.isna(last_impressions)
        ):
            return "Insufficient Data"

        # Lower rank number means a better position
        rank_improved = last_rank < first_rank
        rank_declined = last_rank > first_rank

        impressions_increased = (
            last_impressions >= first_impressions
        )

        impressions_declined = (
            last_impressions <= first_impressions
        )

        if rank_improved and impressions_increased:
            return "Gaining"

        if rank_declined and impressions_declined:
            return "Losing"

        return "Mixed"

    theme_visibility["Visibility_Trend"] = (
        theme_visibility.apply(
            classify_visibility,
            axis=1
        )
    )

    if theme_visibility.empty:
        raise ValueError(
            "Theme visibility analysis returned no results."
        )

    # Readable output
    theme_visibility["First_Avg_Rank"] = (
        theme_visibility["First_Avg_Rank"]
        .round(1)
    )

    theme_visibility["Last_Avg_Rank"] = (
        theme_visibility["Last_Avg_Rank"]
        .round(1)
    )

    theme_visibility["Impressions_Change"] = (
        pd.to_numeric(
            theme_visibility["Impressions_Change"],
            errors="coerce"
        ).round(2)
    )

    theme_visibility["Clicks_Change"] = (
        pd.to_numeric(
            theme_visibility["Clicks_Change"],
            errors="coerce"
        ).round(2)
    )

    output_columns = [
        "Theme",
        "First_Avg_Rank",
        "Last_Avg_Rank",
        "Impressions_Change",
        "Clicks_Change",
        "Visibility_Trend"
    ]

    print(
        f"DONE - Theme visibility analyzed successfully "
        f"from {first_month} to {last_month}."
    )

    print(
        f"INFO - {len(theme_visibility)} theme group(s) analyzed."
    )

    display(
        theme_visibility[output_columns]
    )

except Exception as e:
    print(
        f"FAIL - Theme visibility analysis error: {e}"
    )

DONE - Theme visibility analyzed successfully from 2026-05 to 2026-07.
INFO - 5 theme group(s) analyzed.


,Theme,First_Avg_Rank,Last_Avg_Rank,Impressions_Change,Clicks_Change,Visibility_Trend
0,Demand Planning,30.5,34.5,33.48,35.00,Mixed
1,Freight & Logistics,21.1,24.8,6.41,19.10,Mixed
2,Supply Chain Visibility,26.0,23.6,-36.02,-3.30,Mixed
3,Unmapped,23.5,25.8,-23.10,-31.26,Losing
4,Warehouse Management,29.1,32.0,85.66,44.40,Mixed


## 11. Export Final Output,

In [24]:
# 11. Export and Format Task 1 Output

import os

from openpyxl import load_workbook
from openpyxl.styles import (
    Font,
    PatternFill,
    Alignment,
    Border,
    Side
)
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.chart import BarChart, Reference
from openpyxl.utils import get_column_letter

try:
    task1_output_file = "/content/task1_seo_analysis_output.xlsx"

    # Prepare Merged Data
    merged_export = merged_data.copy()

    merged_columns = [
        "Month_Key",
        "Keyword_Clean",
        "Avg_Rank",
        "Avg_Search_Volume",
        "Rank_Tracker_Clicks",
        "Rank_Tracker_Impressions",
        "Merge_Key",
        "GSC_Clicks",
        "GSC_Impressions",
        "GSC_CTR",
        "GSC_Avg_Position",
        "Theme",
        "Priority",
        "Unmapped_Flag"
    ]

    merged_export = merged_export[merged_columns]

    merged_export["Avg_Rank"] = (
        merged_export["Avg_Rank"].round(1)
    )

    merged_export["Avg_Search_Volume"] = (
        merged_export["Avg_Search_Volume"].round(0)
    )

    merged_export["GSC_Avg_Position"] = (
        merged_export["GSC_Avg_Position"].round(1)
    )

    # Keep CTR as decimal for Excel percentage formatting
    merged_export = merged_export.rename(
        columns={
            "GSC_CTR": "GSC_CTR_%"
        }
    )

    # Prepare Theme Visibility
    theme_export = theme_visibility[
        [
            "Theme",
            "First_Avg_Rank",
            "Last_Avg_Rank",
            "Impressions_Change",
            "Clicks_Change",
            "Visibility_Trend"
        ]
    ].copy()

    # Step 10 stores changes as percentage points.
    # Convert them to decimals for Excel percentage formatting.
    theme_export["Impressions_Change"] = (
        theme_export["Impressions_Change"] / 100
    )

    theme_export["Clicks_Change"] = (
        theme_export["Clicks_Change"] / 100
    )

    theme_export = theme_export.rename(
        columns={
            "Impressions_Change": "Impressions_Change_%",
            "Clicks_Change": "Clicks_Change_%"
        }
    )

    # Prepare Unmapped Keywords
    unmapped_export = (
        merged_data.loc[
            merged_data["Unmapped_Flag"] == "Yes",
            ["Keyword_Clean"]
        ]
        .drop_duplicates()
        .sort_values("Keyword_Clean")
        .reset_index(drop=True)
        .rename(
            columns={
                "Keyword_Clean": "Unmapped_Keyword"
            }
        )
    )

    # Export Task 1 data
    with pd.ExcelWriter(
        task1_output_file,
        engine="openpyxl"
    ) as writer:

        merged_export.to_excel(
            writer,
            sheet_name="Merged_Data",
            index=False
        )

        theme_export.to_excel(
            writer,
            sheet_name="Theme_Visibility",
            index=False
        )

        unmapped_export.to_excel(
            writer,
            sheet_name="Unmapped_Keywords",
            index=False
        )

    # Open workbook for formatting
    wb = load_workbook(task1_output_file)

    header_fill = PatternFill(
        fill_type="solid",
        fgColor="1F4E78"
    )

    header_font = Font(
        color="FFFFFF",
        bold=True
    )

    thin_gray = Side(
        style="thin",
        color="D9E1F2"
    )

    cell_border = Border(
        bottom=thin_gray
    )

    center_alignment = Alignment(
        horizontal="center",
        vertical="center"
    )

    left_alignment = Alignment(
        horizontal="left",
        vertical="center"
    )

    green_fill = PatternFill(
        fill_type="solid",
        fgColor="E2F0D9"
    )

    yellow_fill = PatternFill(
        fill_type="solid",
        fgColor="FFF2CC"
    )

    red_fill = PatternFill(
        fill_type="solid",
        fgColor="FCE4D6"
    )

    gray_fill = PatternFill(
        fill_type="solid",
        fgColor="E7E6E6"
    )

    # Reusable worksheet formatting
    def format_sheet(
        ws,
        freeze_cell="A2",
        add_table=True
    ):
        ws.freeze_panes = freeze_cell
        ws.sheet_view.showGridLines = False

        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = center_alignment

        for row in ws.iter_rows(
            min_row=2,
            max_row=ws.max_row
        ):
            for cell in row:
                cell.border = cell_border
                cell.alignment = left_alignment

        # Adjust column widths
        for column_cells in ws.columns:
            max_length = 0

            column_letter = get_column_letter(
                column_cells[0].column
            )

            for cell in column_cells:
                if cell.value is not None:
                    value_length = len(str(cell.value))
                    max_length = max(
                        max_length,
                        value_length
                    )

            adjusted_width = min(
                max(max_length + 2, 12),
                40
            )

            ws.column_dimensions[
                column_letter
            ].width = adjusted_width

        # Add Excel table
        if (
            add_table
            and ws.max_row >= 2
            and ws.max_column >= 1
        ):
            table_ref = (
                f"A1:"
                f"{get_column_letter(ws.max_column)}"
                f"{ws.max_row}"
            )

            table_name = (
                ws.title
                .replace(" ", "_")
                .replace("-", "_")
                + "_Table"
            )

            table = Table(
                displayName=table_name,
                ref=table_ref
            )

            style = TableStyleInfo(
                name="TableStyleMedium2",
                showFirstColumn=False,
                showLastColumn=False,
                showRowStripes=True,
                showColumnStripes=False
            )

            table.tableStyleInfo = style
            ws.add_table(table)

    # Format Merged_Data
    ws_merged = wb["Merged_Data"]

    format_sheet(
        ws_merged,
        freeze_cell="A2"
    )

    merged_headers = {
        cell.value: cell.column
        for cell in ws_merged[1]
    }

    numeric_formats = {
        "Avg_Rank": "0.0",
        "Avg_Search_Volume": "#,##0",
        "Rank_Tracker_Clicks": "#,##0",
        "Rank_Tracker_Impressions": "#,##0",
        "GSC_Clicks": "#,##0",
        "GSC_Impressions": "#,##0",
        "GSC_CTR_%": "0.00%",
        "GSC_Avg_Position": "0.0"
    }

    for column_name, number_format in numeric_formats.items():

        if column_name in merged_headers:

            column_number = merged_headers[
                column_name
            ]

            for row in range(
                2,
                ws_merged.max_row + 1
            ):
                ws_merged.cell(
                    row=row,
                    column=column_number
                ).number_format = number_format

    # Priority highlighting
    if "Priority" in merged_headers:

        priority_col = merged_headers["Priority"]

        for row in range(
            2,
            ws_merged.max_row + 1
        ):
            cell = ws_merged.cell(
                row=row,
                column=priority_col
            )

            if cell.value == "High":
                cell.fill = green_fill

            elif cell.value == "Medium":
                cell.fill = yellow_fill

            elif cell.value == "Low":
                cell.fill = gray_fill

            elif cell.value == "Unmapped":
                cell.fill = red_fill

    # Unmapped highlighting
    if "Unmapped_Flag" in merged_headers:

        unmapped_col = merged_headers[
            "Unmapped_Flag"
        ]

        for row in range(
            2,
            ws_merged.max_row + 1
        ):
            cell = ws_merged.cell(
                row=row,
                column=unmapped_col
            )

            if cell.value == "Yes":
                cell.fill = red_fill
                cell.font = Font(
                    bold=True
                )

    # Format Theme_Visibility
    ws_theme = wb["Theme_Visibility"]

    format_sheet(
        ws_theme,
        freeze_cell="A2"
    )

    theme_headers = {
        cell.value: cell.column
        for cell in ws_theme[1]
    }

    # Rank formatting
    for column_name in [
        "First_Avg_Rank",
        "Last_Avg_Rank"
    ]:
        column_number = theme_headers[
            column_name
        ]

        for row in range(
            2,
            ws_theme.max_row + 1
        ):
            ws_theme.cell(
                row=row,
                column=column_number
            ).number_format = "0.0"

    # Percentage formatting
    for column_name in [
        "Impressions_Change_%",
        "Clicks_Change_%"
    ]:
        column_number = theme_headers[
            column_name
        ]

        for row in range(
            2,
            ws_theme.max_row + 1
        ):
            ws_theme.cell(
                row=row,
                column=column_number
            ).number_format = "0.00%"

    # Trend highlighting
    trend_col = theme_headers[
        "Visibility_Trend"
    ]

    for row in range(
        2,
        ws_theme.max_row + 1
    ):
        cell = ws_theme.cell(
            row=row,
            column=trend_col
        )

        if cell.value == "Gaining":
            cell.fill = green_fill

        elif cell.value == "Mixed":
            cell.fill = yellow_fill

        elif cell.value == "Losing":
            cell.fill = red_fill

        elif cell.value == "Insufficient Data":
            cell.fill = gray_fill

        cell.font = Font(
            bold=True
        )

    # Add Theme Visibility chart
    chart = BarChart()

    chart.type = "col"
    chart.style = 10

    chart.title = (
        f"Theme Performance: "
        f"{first_month} vs {last_month}"
    )

    chart.y_axis.title = "Change (%)"
    chart.x_axis.title = "Theme"

    data = Reference(
        ws_theme,
        min_col=4,
        max_col=5,
        min_row=1,
        max_row=ws_theme.max_row
    )

    categories = Reference(
        ws_theme,
        min_col=1,
        min_row=2,
        max_row=ws_theme.max_row
    )

    chart.add_data(
        data,
        titles_from_data=True
    )

    chart.set_categories(
        categories
    )

    chart.height = 8
    chart.width = 16

    ws_theme.add_chart(
        chart,
        "H2"
    )

    ws_theme.column_dimensions["H"].width = 3

    # Format Unmapped_Keywords
    ws_unmapped = wb["Unmapped_Keywords"]

    format_sheet(
        ws_unmapped,
        freeze_cell="A2"
    )

    ws_unmapped.column_dimensions[
        "A"
    ].width = 40

    for row in range(
        2,
        ws_unmapped.max_row + 1
    ):
        ws_unmapped.cell(
            row=row,
            column=1
        ).fill = red_fill

    # Save workbook
    wb.save(task1_output_file)

    # Validate output file
    if not os.path.exists(task1_output_file):
        raise FileNotFoundError(
            "The Task 1 output workbook was not created."
        )

    if os.path.getsize(task1_output_file) == 0:
        raise ValueError(
            "The Task 1 output workbook is empty."
        )

    validation_wb = load_workbook(
        task1_output_file,
        read_only=True
    )

    expected_sheets = {
        "Merged_Data",
        "Theme_Visibility",
        "Unmapped_Keywords"
    }

    actual_sheets = set(
        validation_wb.sheetnames
    )

    validation_wb.close()

    missing_sheets = (
        expected_sheets - actual_sheets
    )

    if missing_sheets:
        raise ValueError(
            f"Missing output sheet(s): "
            f"{', '.join(missing_sheets)}"
        )

    print(
        "DONE - Task 1 formatted workbook created successfully."
    )

    print(
        f"INFO - Merged_Data: "
        f"{len(merged_export)} rows."
    )

    print(
        f"INFO - Theme_Visibility: "
        f"{len(theme_export)} theme groups."
    )

    print(
        f"INFO - Unmapped_Keywords: "
        f"{len(unmapped_export)} unique keywords."
    )

    print(
        "INFO - Task 1 output saved for use in the full workflow."
    )

    print(
        f"INFO - Output file: "
        f"{task1_output_file}"
    )

except Exception as e:
    print(
        f"FAIL - Task 1 export or formatting error: {e}"
    )

DONE - Task 1 formatted workbook created successfully.
INFO - Merged_Data: 54 rows.
INFO - Theme_Visibility: 5 theme groups.
INFO - Unmapped_Keywords: 3 unique keywords.
INFO - Task 1 output saved for use in the full workflow.
INFO - Output file: /content/task1_seo_analysis_output.xlsx


# TASK 2 — AUTOMATION BUILD


## 12. Product Theme Monthly Summary

In [25]:
# 12. Product Theme Monthly Summary

try:
    theme_monthly_report = (
        merged_data
        .groupby(
            ["Month_Key", "Theme"],
            as_index=False
        )
        .agg(
            Avg_Rank=("Avg_Rank", "mean"),
            GSC_Impressions=("GSC_Impressions", "sum"),
            GSC_Clicks=("GSC_Clicks", "sum")
        )
        .sort_values(
            ["Theme", "Month_Key"]
        )
        .reset_index(drop=True)
    )

    if theme_monthly_report.empty:
        raise ValueError(
            "Theme monthly report returned no results."
        )

    duplicate_rows = (
        theme_monthly_report[
            ["Month_Key", "Theme"]
        ]
        .duplicated()
        .sum()
    )

    if duplicate_rows > 0:
        raise ValueError(
            f"{duplicate_rows} duplicate Month + Theme row(s) found."
        )

    theme_monthly_report["Avg_Rank"] = (
        theme_monthly_report["Avg_Rank"].round(1)
    )

    print(
        "DONE - Product Theme monthly summary created successfully."
    )

    print(
        f"INFO - {len(theme_monthly_report)} Month + Theme row(s) created."
    )

    print(
        f"INFO - {theme_monthly_report['Theme'].nunique()} theme group(s) found."
    )

    display(theme_monthly_report)

except Exception as e:
    print(
        f"FAIL - Product Theme monthly summary error: {e}"
    )

DONE - Product Theme monthly summary created successfully.
INFO - 15 Month + Theme row(s) created.
INFO - 5 theme group(s) found.


,Month_Key,Theme,Avg_Rank,GSC_Impressions,GSC_Clicks
0,2026-05,Demand Planning,30.5,8070,340
1,2026-06,Demand Planning,32.9,12236,791
2,2026-07,Demand Planning,34.5,10772,459
3,2026-05,Freight & Logistics,21.1,13511,864
4,2026-06,Freight & Logistics,21.5,13216,694
5,2026-07,Freight & Logistics,24.8,14377,1029
6,2026-05,Supply Chain Visibility,26.0,15332,758
7,2026-06,Supply Chain Visibility,26.0,13135,863
8,2026-07,Supply Chain Visibility,23.6,9809,733
9,2026-05,Unmapped,23.5,13465,1030


## 13. Month-over-Month Changes

In [26]:
# 13. Month-over-Month Changes

try:
    theme_mom = theme_monthly_report.copy()

    theme_mom = (
        theme_mom
        .sort_values(["Theme", "Month_Key"])
        .reset_index(drop=True)
    )

    # Previous month values by Theme
    theme_mom["Prev_Avg_Rank"] = (
        theme_mom
        .groupby("Theme")["Avg_Rank"]
        .shift(1)
    )

    theme_mom["Prev_GSC_Impressions"] = (
        theme_mom
        .groupby("Theme")["GSC_Impressions"]
        .shift(1)
    )

    theme_mom["Prev_GSC_Clicks"] = (
        theme_mom
        .groupby("Theme")["GSC_Clicks"]
        .shift(1)
    )

    # Positive = ranking improved
    # Negative = ranking became worse
    theme_mom["Rank_Change"] = (
        theme_mom["Prev_Avg_Rank"]
        - theme_mom["Avg_Rank"]
    )

    def safe_mom_change(current_value, previous_value):

        if (
            pd.isna(current_value)
            or pd.isna(previous_value)
            or previous_value == 0
        ):
            return pd.NA

        return (
            (current_value - previous_value)
            / previous_value
            * 100
        )

    theme_mom["Impressions_MoM_%"] = (
        theme_mom.apply(
            lambda row: safe_mom_change(
                row["GSC_Impressions"],
                row["Prev_GSC_Impressions"]
            ),
            axis=1
        )
    )

    theme_mom["Clicks_MoM_%"] = (
        theme_mom.apply(
            lambda row: safe_mom_change(
                row["GSC_Clicks"],
                row["Prev_GSC_Clicks"]
            ),
            axis=1
        )
    )

    # Readable values
    theme_mom["Rank_Change"] = (
        theme_mom["Rank_Change"].round(1)
    )

    theme_mom["Impressions_MoM_%"] = (
        pd.to_numeric(
            theme_mom["Impressions_MoM_%"],
            errors="coerce"
        ).round(2)
    )

    theme_mom["Clicks_MoM_%"] = (
        pd.to_numeric(
            theme_mom["Clicks_MoM_%"],
            errors="coerce"
        ).round(2)
    )

    if theme_mom.empty:
        raise ValueError(
            "Month-over-month report returned no results."
        )

    print(
        "DONE - Month-over-month changes calculated successfully."
    )

    print(
        "INFO - Positive Rank_Change means ranking improved; "
        "negative means ranking became worse."
    )

    display(
        theme_mom[
            [
                "Month_Key",
                "Theme",
                "Avg_Rank",
                "GSC_Impressions",
                "GSC_Clicks",
                "Rank_Change",
                "Impressions_MoM_%",
                "Clicks_MoM_%"
            ]
        ]
    )

except Exception as e:
    print(
        f"FAIL - Month-over-month calculation error: {e}"
    )

DONE - Month-over-month changes calculated successfully.
INFO - Positive Rank_Change means ranking improved; negative means ranking became worse.


,Month_Key,Theme,Avg_Rank,GSC_Impressions,GSC_Clicks,Rank_Change,Impressions_MoM_%,Clicks_MoM_%
0,2026-05,Demand Planning,30.5,8070,340,NaN,NaN,NaN
1,2026-06,Demand Planning,32.9,12236,791,-2.4,51.62,132.65
2,2026-07,Demand Planning,34.5,10772,459,-1.6,-11.96,-41.97
3,2026-05,Freight & Logistics,21.1,13511,864,NaN,NaN,NaN
4,2026-06,Freight & Logistics,21.5,13216,694,-0.4,-2.18,-19.68
5,2026-07,Freight & Logistics,24.8,14377,1029,-3.3,8.78,48.27
6,2026-05,Supply Chain Visibility,26.0,15332,758,NaN,NaN,NaN
7,2026-06,Supply Chain Visibility,26.0,13135,863,0.0,-14.33,13.85
8,2026-07,Supply Chain Visibility,23.6,9809,733,2.4,-25.32,-15.06
9,2026-05,Unmapped,23.5,13465,1030,NaN,NaN,NaN


## 14. Keywords Needing Attention

In [27]:
# 14. Keywords Needing Attention

try:
    RANK_DROP_THRESHOLD = 3
    CTR_DROP_THRESHOLD = 20

    keyword_attention = (
        merged_data
        .copy()
        .sort_values(
            ["Keyword_Clean", "Month_Key"]
        )
        .reset_index(drop=True)
    )

    # Previous month values
    keyword_attention["Prev_Avg_Rank"] = (
        keyword_attention
        .groupby("Keyword_Clean")["Avg_Rank"]
        .shift(1)
    )

    keyword_attention["Prev_GSC_CTR"] = (
        keyword_attention
        .groupby("Keyword_Clean")["GSC_CTR"]
        .shift(1)
    )

    # Ranking change
    # Positive = improvement
    # Negative = decline
    keyword_attention["Rank_Change"] = (
        keyword_attention["Prev_Avg_Rank"]
        - keyword_attention["Avg_Rank"]
    )

    # Safe CTR month-over-month change
    def safe_ctr_change(current_ctr, previous_ctr):

        if (
            pd.isna(current_ctr)
            or pd.isna(previous_ctr)
            or previous_ctr == 0
        ):
            return pd.NA

        return (
            (current_ctr - previous_ctr)
            / previous_ctr
            * 100
        )

    keyword_attention["CTR_MoM_%"] = (
        keyword_attention.apply(
            lambda row: safe_ctr_change(
                row["GSC_CTR"],
                row["Prev_GSC_CTR"]
            ),
            axis=1
        )
    )

    keyword_attention["CTR_MoM_%"] = (
        pd.to_numeric(
            keyword_attention["CTR_MoM_%"],
            errors="coerce"
        )
    )

    # Attention rules
    keyword_attention["Ranking_Drop_Flag"] = (
        keyword_attention["Rank_Change"]
        <= -RANK_DROP_THRESHOLD
    )

    keyword_attention["CTR_Drop_Flag"] = (
        keyword_attention["CTR_MoM_%"]
        <= -CTR_DROP_THRESHOLD
    )

    keyword_attention["Missing_Mapping_Flag"] = (
        keyword_attention["Unmapped_Flag"] == "Yes"
    )

    def attention_reason(row):
        reasons = []

        if row["Ranking_Drop_Flag"]:
            reasons.append("Ranking Drop")

        if row["CTR_Drop_Flag"]:
            reasons.append("CTR Drop")

        if row["Missing_Mapping_Flag"]:
            reasons.append("Missing Mapping")

        return ", ".join(reasons)

    keyword_attention["Attention_Reason"] = (
        keyword_attention.apply(
            attention_reason,
            axis=1
        )
    )

    # Keep full alert history
    keyword_attention_history = (
        keyword_attention.loc[
            keyword_attention["Attention_Reason"] != "",
            [
                "Month_Key",
                "Keyword_Clean",
                "Theme",
                "Priority",
                "Avg_Rank",
                "Prev_Avg_Rank",
                "Rank_Change",
                "GSC_CTR",
                "Prev_GSC_CTR",
                "CTR_MoM_%",
                "Attention_Reason"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    # Current reporting period
    latest_month = (
        keyword_attention["Month_Key"]
        .dropna()
        .max()
    )

    if pd.isna(latest_month):
        raise ValueError(
            "No valid reporting month was found."
        )

    keywords_needing_attention = (
        keyword_attention_history[
            keyword_attention_history["Month_Key"]
            == latest_month
        ]
        .copy()
        .reset_index(drop=True)
    )

    # Readable values
    for df in [
        keyword_attention_history,
        keywords_needing_attention
    ]:
        df["Avg_Rank"] = (
            df["Avg_Rank"].round(1)
        )

        df["Prev_Avg_Rank"] = (
            df["Prev_Avg_Rank"].round(1)
        )

        df["Rank_Change"] = (
            df["Rank_Change"].round(1)
        )

        df["CTR_MoM_%"] = (
            df["CTR_MoM_%"].round(2)
        )

    print(
        "DONE - Keyword attention analysis completed successfully."
    )

    print(
        f"INFO - Latest month detected: {latest_month}."
    )

    print(
        f"INFO - {len(keyword_attention_history)} "
        "historical Month + Keyword alert row(s) found."
    )

    print(
        f"INFO - {len(keywords_needing_attention)} "
        "keyword(s) currently need attention."
    )

    print(
        f"INFO - Ranking Drop rule: "
        f"{RANK_DROP_THRESHOLD}+ positions."
    )

    print(
        f"INFO - CTR Drop rule: "
        f"{CTR_DROP_THRESHOLD}%+ MoM decline."
    )

    print(
        "\nCURRENT KEYWORDS NEEDING ATTENTION"
    )

    display(
        keywords_needing_attention
    )

except Exception as e:
    print(
        f"FAIL - Keyword attention analysis error: {e}"
    )

DONE - Keyword attention analysis completed successfully.
INFO - Latest month detected: 2026-07.
INFO - 22 historical Month + Keyword alert row(s) found.
INFO - 10 keyword(s) currently need attention.
INFO - Ranking Drop rule: 3+ positions.
INFO - CTR Drop rule: 20%+ MoM decline.

CURRENT KEYWORDS NEEDING ATTENTION


,Month_Key,Keyword_Clean,Theme,Priority,Avg_Rank,Prev_Avg_Rank,Rank_Change,GSC_CTR,Prev_GSC_CTR,CTR_MoM_%,Attention_Reason
0,2026-07,ai demand forecasting platform,Demand Planning,High,35.0,32.1,-2.9,0.020987,0.069585,-69.84,CTR Drop
1,2026-07,cloud based warehouse management software,Warehouse Management,Low,17.0,14.9,-2.1,0.037387,0.077756,-51.92,CTR Drop
2,2026-07,end to end supply chain visibility platform,Supply Chain Visibility,Medium,27.0,24.0,-3.0,0.079040,0.032396,143.98,Ranking Drop
3,2026-07,freight rate benchmarking tool,Freight & Logistics,Medium,22.0,18.8,-3.2,0.037157,0.025913,43.39,Ranking Drop
4,2026-07,logistics automation software,Unmapped,Unmapped,34.0,32.3,-1.7,0.077041,0.090316,-14.70,Missing Mapping
5,2026-07,real time supply chain tracking,Supply Chain Visibility,High,7.0,19.8,12.8,0.075649,0.094567,-20.01,CTR Drop
6,2026-07,supply chain risk management,Unmapped,Unmapped,12.0,12.1,0.1,0.065844,0.068520,-3.91,Missing Mapping
7,2026-07,third party logistics software,Unmapped,Unmapped,31.5,31.1,-0.4,0.062452,0.060549,3.14,Missing Mapping
8,2026-07,transportation management system,Freight & Logistics,High,27.0,18.6,-8.4,0.098486,0.101890,-3.34,Ranking Drop
9,2026-07,wms software for distribution centers,Warehouse Management,Low,47.0,43.8,-3.2,0.012713,0.114603,-88.91,"Ranking Drop, CTR Drop"


## 15. Validate Automated Outputs

In [28]:
# 15. Validate Automated Outputs

try:
    validation_errors = []

    # Validate Product Theme Summary
    required_theme_columns = [
        "Month_Key",
        "Theme",
        "Avg_Rank",
        "GSC_Impressions",
        "GSC_Clicks",
        "Rank_Change",
        "Impressions_MoM_%",
        "Clicks_MoM_%"
    ]

    missing_theme_columns = [
        column
        for column in required_theme_columns
        if column not in theme_mom.columns
    ]

    if missing_theme_columns:
        validation_errors.append(
            "Product Theme Summary is missing column(s): "
            + ", ".join(missing_theme_columns)
        )

    if theme_mom.empty:
        validation_errors.append(
            "Product Theme Summary is empty."
        )

    theme_duplicates = (
        theme_mom[
            ["Month_Key", "Theme"]
        ]
        .duplicated()
        .sum()
    )

    if theme_duplicates > 0:
        validation_errors.append(
            f"Product Theme Summary contains "
            f"{theme_duplicates} duplicate Month + Theme row(s)."
        )

    # Validate Keywords Needing Attention
    required_attention_columns = [
        "Month_Key",
        "Keyword_Clean",
        "Theme",
        "Priority",
        "Avg_Rank",
        "Prev_Avg_Rank",
        "Rank_Change",
        "GSC_CTR",
        "Prev_GSC_CTR",
        "CTR_MoM_%",
        "Attention_Reason"
    ]

    missing_attention_columns = [
        column
        for column in required_attention_columns
        if column not in keywords_needing_attention.columns
    ]

    if missing_attention_columns:
        validation_errors.append(
            "Keywords Needing Attention is missing column(s): "
            + ", ".join(missing_attention_columns)
        )

    attention_duplicates = (
        keywords_needing_attention[
            ["Month_Key", "Keyword_Clean"]
        ]
        .duplicated()
        .sum()
    )

    if attention_duplicates > 0:
        validation_errors.append(
            f"Keywords Needing Attention contains "
            f"{attention_duplicates} duplicate Month + Keyword row(s)."
        )

    # Validate latest reporting period
    latest_month = (
        merged_data["Month_Key"]
        .dropna()
        .max()
    )

    if pd.isna(latest_month):
        validation_errors.append(
            "No valid reporting month was found."
        )

    elif not keywords_needing_attention.empty:

        invalid_month_rows = (
            keywords_needing_attention["Month_Key"]
            != latest_month
        ).sum()

        if invalid_month_rows > 0:
            validation_errors.append(
                f"{invalid_month_rows} attention row(s) "
                f"do not belong to latest month {latest_month}."
            )

    # Validate attention reasons
    valid_reasons = {
        "Ranking Drop",
        "CTR Drop",
        "Missing Mapping"
    }

    invalid_reasons = []

    for reason_text in (
        keywords_needing_attention[
            "Attention_Reason"
        ].dropna()
    ):
        reasons = {
            reason.strip()
            for reason in reason_text.split(",")
        }

        if not reasons.issubset(valid_reasons):
            invalid_reasons.append(reason_text)

    if invalid_reasons:
        validation_errors.append(
            "Unexpected Attention_Reason value(s) found: "
            + ", ".join(
                sorted(set(invalid_reasons))
            )
        )

    # Confirm that all current unmapped keywords were flagged
    current_unmapped = (
        merged_data[
            (merged_data["Month_Key"] == latest_month)
            & (merged_data["Unmapped_Flag"] == "Yes")
        ]["Keyword_Clean"]
        .drop_duplicates()
        .tolist()
    )

    flagged_missing_mapping = (
        keywords_needing_attention[
            keywords_needing_attention[
                "Attention_Reason"
            ].str.contains(
                "Missing Mapping",
                na=False
            )
        ]["Keyword_Clean"]
        .drop_duplicates()
        .tolist()
    )

    missing_mapping_alerts = (
        set(current_unmapped)
        - set(flagged_missing_mapping)
    )

    if missing_mapping_alerts:
        validation_errors.append(
            "Unmapped keyword(s) missing from attention report: "
            + ", ".join(
                sorted(missing_mapping_alerts)
            )
        )

    if validation_errors:
        raise ValueError(
            "\n".join(validation_errors)
        )

    print(
        "DONE - Task 2 automated outputs validated successfully."
    )

    print(
        f"INFO - Latest reporting month: {latest_month}."
    )

    print(
        f"INFO - Product Theme Summary: "
        f"{len(theme_mom)} Month + Theme row(s)."
    )

    print(
        f"INFO - Current Keywords Needing Attention: "
        f"{len(keywords_needing_attention)} keyword(s)."
    )

    print(
        f"INFO - Current Missing Mappings: "
        f"{len(current_unmapped)} keyword(s)."
    )

    print(
        "INFO - No duplicate rows or invalid "
        "attention reasons detected."
    )

except Exception as e:
    print(
        f"FAIL - Task 2 output validation error: {e}"
    )

DONE - Task 2 automated outputs validated successfully.
INFO - Latest reporting month: 2026-07.
INFO - Product Theme Summary: 15 Month + Theme row(s).
INFO - Current Keywords Needing Attention: 10 keyword(s).
INFO - Current Missing Mappings: 3 keyword(s).
INFO - No duplicate rows or invalid attention reasons detected.


## 16. Export Automated Weekly Report

In [29]:
# 16. Export Automated Weekly Report

from google.colab import files
import os

from openpyxl import load_workbook
from openpyxl.styles import (
    Font,
    PatternFill,
    Alignment,
    Border,
    Side
)
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter

try:
    task2_output_file = "/content/task2_weekly_seo_report.xlsx"

    # Prepare Product Theme Summary
    theme_report_export = theme_mom[
        [
            "Month_Key",
            "Theme",
            "Avg_Rank",
            "GSC_Impressions",
            "GSC_Clicks",
            "Rank_Change",
            "Impressions_MoM_%",
            "Clicks_MoM_%"
        ]
    ].copy()

    theme_report_export["Avg_Rank"] = (
        theme_report_export["Avg_Rank"].round(1)
    )

    theme_report_export["Rank_Change"] = (
        theme_report_export["Rank_Change"].round(1)
    )

    # Convert percentage points to decimals for Excel
    theme_report_export["Impressions_MoM_%"] = (
        theme_report_export["Impressions_MoM_%"] / 100
    )

    theme_report_export["Clicks_MoM_%"] = (
        theme_report_export["Clicks_MoM_%"] / 100
    )

    # Prepare current Keywords Needing Attention
    attention_export = keywords_needing_attention.copy()

    attention_export["GSC_CTR"] = (
        attention_export["GSC_CTR"].astype(float)
    )

    attention_export["Prev_GSC_CTR"] = (
        attention_export["Prev_GSC_CTR"].astype(float)
    )

    attention_export["CTR_MoM_%"] = (
        attention_export["CTR_MoM_%"] / 100
    )

    # Prepare historical attention data
    attention_history_export = (
        keyword_attention_history.copy()
    )

    attention_history_export["GSC_CTR"] = (
        attention_history_export["GSC_CTR"].astype(float)
    )

    attention_history_export["Prev_GSC_CTR"] = (
        attention_history_export["Prev_GSC_CTR"].astype(float)
    )

    attention_history_export["CTR_MoM_%"] = (
        attention_history_export["CTR_MoM_%"] / 100
    )

    # Export sheets
    with pd.ExcelWriter(
        task2_output_file,
        engine="openpyxl"
    ) as writer:

        theme_report_export.to_excel(
            writer,
            sheet_name="Product_Theme_Summary",
            index=False
        )

        attention_export.to_excel(
            writer,
            sheet_name="Keywords_Attention",
            index=False
        )

        attention_history_export.to_excel(
            writer,
            sheet_name="Attention_History",
            index=False
        )

    # Open workbook for formatting
    wb = load_workbook(task2_output_file)

    header_fill = PatternFill(
        fill_type="solid",
        fgColor="1F4E78"
    )

    header_font = Font(
        color="FFFFFF",
        bold=True
    )

    thin_gray = Side(
        style="thin",
        color="D9E1F2"
    )

    cell_border = Border(
        bottom=thin_gray
    )

    green_fill = PatternFill(
        fill_type="solid",
        fgColor="E2F0D9"
    )

    yellow_fill = PatternFill(
        fill_type="solid",
        fgColor="FFF2CC"
    )

    red_fill = PatternFill(
        fill_type="solid",
        fgColor="FCE4D6"
    )

    gray_fill = PatternFill(
        fill_type="solid",
        fgColor="E7E6E6"
    )

    center_alignment = Alignment(
        horizontal="center",
        vertical="center"
    )

    left_alignment = Alignment(
        horizontal="left",
        vertical="center"
    )

    # Reusable worksheet formatting
    def format_sheet(ws):

        ws.freeze_panes = "A2"
        ws.sheet_view.showGridLines = False

        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = center_alignment

        for row in ws.iter_rows(
            min_row=2,
            max_row=ws.max_row
        ):
            for cell in row:
                cell.border = cell_border
                cell.alignment = left_alignment

        # Adjust column widths
        for column_cells in ws.columns:

            max_length = 0

            column_letter = get_column_letter(
                column_cells[0].column
            )

            for cell in column_cells:

                if cell.value is not None:
                    max_length = max(
                        max_length,
                        len(str(cell.value))
                    )

            ws.column_dimensions[
                column_letter
            ].width = min(
                max(max_length + 2, 12),
                42
            )

        # Add Excel table
        if ws.max_row >= 2:

            table_ref = (
                f"A1:"
                f"{get_column_letter(ws.max_column)}"
                f"{ws.max_row}"
            )

            table_name = (
                ws.title
                .replace(" ", "_")
                .replace("-", "_")
                + "_Table"
            )

            table = Table(
                displayName=table_name,
                ref=table_ref
            )

            style = TableStyleInfo(
                name="TableStyleMedium2",
                showFirstColumn=False,
                showLastColumn=False,
                showRowStripes=True,
                showColumnStripes=False
            )

            table.tableStyleInfo = style
            ws.add_table(table)

    # Product Theme Summary
    ws_theme = wb["Product_Theme_Summary"]

    format_sheet(ws_theme)

    theme_headers = {
        cell.value: cell.column
        for cell in ws_theme[1]
    }

    for row in range(
        2,
        ws_theme.max_row + 1
    ):

        ws_theme.cell(
            row=row,
            column=theme_headers["Avg_Rank"]
        ).number_format = "0.0"

        ws_theme.cell(
            row=row,
            column=theme_headers["GSC_Impressions"]
        ).number_format = "#,##0"

        ws_theme.cell(
            row=row,
            column=theme_headers["GSC_Clicks"]
        ).number_format = "#,##0"

        ws_theme.cell(
            row=row,
            column=theme_headers["Rank_Change"]
        ).number_format = "0.0"

        ws_theme.cell(
            row=row,
            column=theme_headers["Impressions_MoM_%"]
        ).number_format = "0.00%"

        ws_theme.cell(
            row=row,
            column=theme_headers["Clicks_MoM_%"]
        ).number_format = "0.00%"

    # Highlight Rank Change
    rank_change_col = theme_headers["Rank_Change"]

    for row in range(
        2,
        ws_theme.max_row + 1
    ):

        cell = ws_theme.cell(
            row=row,
            column=rank_change_col
        )

        if isinstance(cell.value, (int, float)):

            if cell.value > 0:
                cell.fill = green_fill

            elif cell.value < 0:
                cell.fill = red_fill

            else:
                cell.fill = gray_fill

    # Keywords Needing Attention
    ws_attention = wb["Keywords_Attention"]

    format_sheet(ws_attention)

    attention_headers = {
        cell.value: cell.column
        for cell in ws_attention[1]
    }

    for row in range(
        2,
        ws_attention.max_row + 1
    ):

        for col_name in [
            "Avg_Rank",
            "Prev_Avg_Rank",
            "Rank_Change"
        ]:
            ws_attention.cell(
                row=row,
                column=attention_headers[col_name]
            ).number_format = "0.0"

        for col_name in [
            "GSC_CTR",
            "Prev_GSC_CTR",
            "CTR_MoM_%"
        ]:
            ws_attention.cell(
                row=row,
                column=attention_headers[col_name]
            ).number_format = "0.00%"

    # Highlight attention reasons
    reason_col = attention_headers[
        "Attention_Reason"
    ]

    for row in range(
        2,
        ws_attention.max_row + 1
    ):

        cell = ws_attention.cell(
            row=row,
            column=reason_col
        )

        reason = str(
            cell.value or ""
        )

        if "," in reason:
            cell.fill = red_fill
            cell.font = Font(bold=True)

        elif "Missing Mapping" in reason:
            cell.fill = red_fill

        elif "Ranking Drop" in reason:
            cell.fill = yellow_fill

        elif "CTR Drop" in reason:
            cell.fill = yellow_fill

    # Priority highlighting
    if "Priority" in attention_headers:

        priority_col = attention_headers[
            "Priority"
        ]

        for row in range(
            2,
            ws_attention.max_row + 1
        ):

            cell = ws_attention.cell(
                row=row,
                column=priority_col
            )

            if cell.value == "High":
                cell.fill = green_fill

            elif cell.value == "Medium":
                cell.fill = yellow_fill

            elif cell.value == "Low":
                cell.fill = gray_fill

            elif cell.value == "Unmapped":
                cell.fill = red_fill

    # Attention History
    ws_history = wb["Attention_History"]

    format_sheet(ws_history)

    history_headers = {
        cell.value: cell.column
        for cell in ws_history[1]
    }

    for row in range(
        2,
        ws_history.max_row + 1
    ):

        for col_name in [
            "Avg_Rank",
            "Prev_Avg_Rank",
            "Rank_Change"
        ]:
            ws_history.cell(
                row=row,
                column=history_headers[col_name]
            ).number_format = "0.0"

        for col_name in [
            "GSC_CTR",
            "Prev_GSC_CTR",
            "CTR_MoM_%"
        ]:
            ws_history.cell(
                row=row,
                column=history_headers[col_name]
            ).number_format = "0.00%"

    # Save workbook
    wb.save(task2_output_file)

    # Validate workbook
    if not os.path.exists(task2_output_file):
        raise FileNotFoundError(
            "The Task 2 output workbook was not created."
        )

    if os.path.getsize(task2_output_file) == 0:
        raise ValueError(
            "The Task 2 output workbook is empty."
        )

    validation_wb = load_workbook(
        task2_output_file,
        read_only=True
    )

    expected_sheets = {
        "Product_Theme_Summary",
        "Keywords_Attention",
        "Attention_History"
    }

    actual_sheets = set(
        validation_wb.sheetnames
    )

    validation_wb.close()

    missing_sheets = (
        expected_sheets - actual_sheets
    )

    if missing_sheets:
        raise ValueError(
            "Missing output sheet(s): "
            + ", ".join(
                sorted(missing_sheets)
            )
        )

    print(
        "DONE - Task 2 automated weekly report created successfully."
    )

    print(
        f"INFO - Product Theme Summary: "
        f"{len(theme_report_export)} row(s)."
    )

    print(
        f"INFO - Current Keywords Needing Attention: "
        f"{len(attention_export)} keyword(s)."
    )

    print(
        f"INFO - Attention History: "
        f"{len(attention_history_export)} alert row(s)."
    )

    print(
        f"INFO - Latest reporting period: "
        f"{latest_month}."
    )

    print(
        f"INFO - Output file: "
        f"{task2_output_file}"
    )

    files.download(task2_output_file)

except Exception as e:
    print(
        f"FAIL - Task 2 export error: {e}"
    )

DONE - Task 2 automated weekly report created successfully.
INFO - Product Theme Summary: 15 row(s).
INFO - Current Keywords Needing Attention: 10 keyword(s).
INFO - Attention History: 22 alert row(s).
INFO - Latest reporting period: 2026-07.
INFO - Output file: /content/task2_weekly_seo_report.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>